In [20]:
import pandas as pd

# Load the first CSV into a DataFrame
stats_df = pd.read_csv("../basketballPlayoffs/players_teams.csv")  # Replace 'stats.csv' with the actual filename of the first CSV

# Load the second CSV into a DataFrame
bio_df = pd.read_csv("../basketballPlayoffs/players.csv")  # Replace 'bio.csv' with the actual filename of the second CSV

# Merge the two DataFrames on playerID and bioID, keeping all data from the first DataFrame
merged_df = stats_df.merge(bio_df[['bioID', 'pos']], left_on='playerID', right_on='bioID', how='left')

# Drop the redundant 'bioID' column after the merge if desired
merged_df.drop(columns=['bioID'], inplace=True)

# Save the merged DataFrame to a new CSV file
merged_df.to_csv('merged_stats.csv', index=False)

print("Merge completed and saved to 'merged_stats.csv'.")


Merge completed and saved to 'merged_stats.csv'.


In [21]:
import pandas as pd

# Load datasets
data = pd.read_csv("merged_stats.csv")
awards = pd.read_csv("../cleaned_datasets/awards_players_cleaned.csv")

# Add missed shot metrics
data['fgMissed'] = data['fgAttempted'] - data['fgMade']
data['ftMissed'] = data['ftAttempted'] - data['ftMade']
data['threeMissed'] = data['threeAttempted'] - data['threeMade']

# Merge awards data into the main dataset
data = data.merge(awards, on=['playerID', 'year'], how='left')

# Award values
award_values = {
    "Most Valuable Player": 10,
    "WNBA Finals Most Valuable Player": 9,
    "Coach of the Year": 1,
    "Defensive Player of the Year": 7,
    "All-Star Game Most Valuable Player": 6,
    "WNBA All-Decade Team": 5,
    "Most Improved Player": 4,
    "WNBA All Decade Team Honorable Mention": 3,
    "Rookie of the Year": 2,
    "Sixth Woman of the Year": 1,
    "Kim Perrot Sportsmanship Award": 0,  # irrelevant for winning
}

# Add award score column
data['award_score'] = data['award'].apply(lambda x: award_values.get(x, 0))

# Sum up total award score for each player per year
award_scores = awards.groupby(['playerID', 'year'])['award'].apply(
    lambda x: sum(award_values.get(award, 0) for award in x)
).reset_index(name='total_award_score')

# Merge total award scores back into the main dataset
data = data.merge(award_scores, on=['playerID', 'year'], how='left')
data['total_award_score'] = data['total_award_score'].fillna(0)

# Define weights for each position
position_weights = {
    'G': {
        'positive': {'points': 0.4, 'assists': 0.3, 'steals': 0.2, 'threeMade': 0.2, 'minutes': 0.1},
        'negative': {'turnovers': -0.3, 'dq': -0.5, 'PF': -0.2, 'threeMissed': -0.1}
    },
    'F': {
        'positive': {'points': 0.3, 'rebounds': 0.4, 'assists': 0.2, 'blocks': 0.2, 'minutes': 0.1},
        'negative': {'turnovers': -0.2, 'dq': -0.5, 'PF': -0.2, 'fgMissed': -0.1}
    },
    'C': {
        'positive': {'rebounds': 0.5, 'blocks': 0.3, 'oRebounds': 0.2, 'points': 0.2, 'minutes': 0.1},
        'negative': {'turnovers': -0.2, 'dq': -0.5, 'PF': -0.2, 'ftMissed': -0.1}
    },
    'F-C': {
        'positive': {'points': 0.26, 'rebounds': 0.46, 'assists': 0.16, 'blocks': 0.26, 'minutes': 0.1},
        'negative': {'turnovers': -0.25, 'dq': -0.5, 'PF': -0.2, 'ftMissed': -0.06}
    },
    'F-G': {
        'positive': {'points': 0.35, 'rebounds': 0.35, 'assists': 0.25, 'steals': 0.15, 'minutes': 0.1},
        'negative': {'turnovers': -0.275, 'dq': -0.5, 'PF': -0.2, 'threeMissed': -0.075}
    },
    'G-F': {
        'positive': {'points': 0.38, 'rebounds': 0.24, 'assists': 0.27, 'steals': 0.18, 'threeMade': 0.18, 'minutes': 0.1},
        'negative': {'turnovers': -0.285, 'dq': -0.5, 'PF': -0.2, 'threeMissed': -0.09}
    }
}

# Function to calculate score based on position and awards
def calculate_position_score(row, position_weights):
    if row['pos'] not in position_weights:
        return None  # Skip if position is not defined in weights
    weights = position_weights[row['pos']]
    positive_score = sum(row[field] * weight for field, weight in weights['positive'].items() if field in row)
    negative_penalty = sum(row[field] * weight for field, weight in weights['negative'].items() if field in row)
    return positive_score + negative_penalty + row['total_award_score']

# Apply score calculation
data['performance_score'] = data.apply(lambda row: calculate_position_score(row, position_weights), axis=1)

# Rank players by position and performance score
ranked_players = data.sort_values(by=['pos', 'performance_score'], ascending=[True, False])

# Display top players for each position
for pos in position_weights.keys():
    print(f"Top Players for Position {pos}:")
    print(ranked_players[ranked_players['pos'] == pos][['playerID', 'performance_score']].head(5))
    print()


Top Players for Position G:
        playerID  performance_score
1285  pondeca01w              379.5
1284  pondeca01w              374.3
1282  pondeca01w              362.8
610   hammobe01w              361.2
609   hammobe01w              357.7

Top Players for Position F:
        playerID  performance_score
1231  parkeca01w              409.8
1232  parkeca01w              409.8
279   catchta01w              386.0
278   catchta01w              375.6
1660  thompti01w              366.6

Top Players for Position C:
       playerID  performance_score
912  leslili01w              404.6
913  leslili01w              404.6
915  leslili01w              391.8
916  leslili01w              391.8
906  leslili01w              384.8

Top Players for Position F-C:
        playerID  performance_score
737   jacksla01w             437.44
738   jacksla01w             437.44
733   jacksla01w             429.29
735   jacksla01w             405.79
1827  willina01w             363.67

Top Players for Position

In [22]:
# Assign year weights (recent years get higher weights)
data['weight'] = data['year'] / 10

# Calculate weighted score for each player
data['weighted_score'] = data['performance_score'] * data['weight']

# Group by playerID and calculate total score
player_scores = data.groupby('playerID').apply(
    lambda group: group['weighted_score'].sum() / group['weight'].sum()
).reset_index(name='total_score')

# Round the total_score to 1 decimal place
player_scores['total_score'] = player_scores['total_score'].round(1)

# Save the results to a CSV file
output_file = 'player_performance_scores.csv'
player_scores.to_csv(output_file, index=False)

print(f"Player performance scores saved to {output_file}")

Player performance scores saved to player_performance_scores.csv


In [ ]:
# Assign year weights (recent years get higher weights)
data['weight'] = data['year'] / 10

# Calculate weighted score for each player
data['weighted_score'] = data['performance_score'] * data['weight']

# Group by playerID and calculate total score
player_scores = data.groupby('playerID').apply(
    lambda group: pd.Series({
        'total_score': (group['weighted_score'].sum() / group['weight'].sum()).round(1),
        'years': ','.join(map(str, group['year'].unique()))  # Include years as a comma-separated list
    })
).reset_index()

# Save the results to a CSV file
output_file = 'player_performance_scores.csv'
player_scores.to_csv(output_file, index=False)

print(f"Player performance scores saved to {output_file}")
